In [2]:
# Install library "Synthetic Data Vault (SDV)", a Python library for creating tabular synthetic data.

!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.0/207.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 8.1 MB/s eta 0:00:00


In [5]:
import numpy as np
import pandas as pd
import os

def generate_statistical_and_rule_based_data(num_samples=1000):
    """
    Hands-On #1: Generating synthetic data using Statistical distributions
    and enforcing business logic via Rule-Based generation.
    """
    print(f"Generating {num_samples} samples of synthetic data...")

    # ---------------------------------------------------------
    # STEP 1: Statistical Generation (Random Sampling)
    # ---------------------------------------------------------
    # Set a random seed for reproducibility
    np.random.seed(42)

    # Generate Age: Normal distribution centered around 35, std dev of 10
    # Clipped between 18 and 80 to ensure realistic adult ages
    age = np.random.normal(loc=35, scale=10, size=num_samples)
    age = np.clip(age, 18, 80).astype(int)

    # Generate Annual Income: Lognormal distribution (right-skewed like real wealth)
    income = np.random.lognormal(mean=10.8, sigma=0.5, size=num_samples)
    income = np.round(income, 2)

    # Generate Credit Score: Uniform distribution between 300 and 850
    credit_score = np.random.randint(low=300, high=850, size=num_samples)

    # ---------------------------------------------------------
    # STEP 2: Rule-Based Generation (Business Logic)
    # ---------------------------------------------------------
    # We want to predict 'Loan_Approved' (1 for Yes, 0 for No)
    # We apply specific rules to determine the base label

    loan_approved = np.zeros(num_samples, dtype=int)

    for i in range(num_samples):
        # Rule 1: High credit score and decent income = Approved
        if credit_score[i] > 700 and income[i] > 40000:
            loan_approved[i] = 1
        # Rule 2: Moderate credit score but high income = Approved
        elif credit_score[i] > 600 and income[i] > 70000:
            loan_approved[i] = 1
        # Rule 3: Very young applicants with low income = Rejected
        elif age[i] < 21 and income[i] < 30000:
            loan_approved[i] = 0
        # Otherwise: Randomly approve 30% of the remaining to introduce noise
        else:
            loan_approved[i] = np.random.choice([0, 1], p=[0.7, 0.3])

    # ---------------------------------------------------------
    # STEP 3: Assemble and Save
    # ---------------------------------------------------------
    df = pd.DataFrame({
        'Age': age,
        'Annual_Income': income,
        'Credit_Score': credit_score,
        'Loan_Approved': loan_approved
    })

    # Create directory if it doesn't exist
    os.makedirs('data', exist_ok=True)
    file_path = 'data/synthetic_loan_data_rules.csv'
    df.to_csv(file_path, index=False)

    print(f"Data successfully generated and saved to '{file_path}'")
    print("Sample Data:")
    print(df.head())

if __name__ == "__main__":
    generate_statistical_and_rule_based_data()

Generating 1000 samples of synthetic data...
Data successfully generated and saved to 'data/synthetic_loan_data_rules.csv'
Sample Data:
   Age  Annual_Income  Credit_Score  Loan_Approved
0   39       98683.96           713              1
1   33       77832.69           657              1
2   41       50504.37           374              0
3   50       35473.16           654              0
4   32       69502.06           487              0


In [6]:
import pandas as pd
import os
import warnings
warnings.filterwarnings("ignore")

# Note: For this hands-on, you must install the SDV library
# Run: pip install sdv

def generate_genai_data():
    """
    Hands-On #2: Generating synthetic data using Generative AI (CTGAN).
    This script reads the rule-based data generated in Hands-On #1,
    learns its multi-dimensional distributions, and generates completely
    new, structurally identical synthetic data.
    """
    try:
        from sdv.single_table import CTGANSynthesizer
        from sdv.metadata import SingleTableMetadata
    except ImportError:
        print("SDV library not found. Please install it using: pip install sdv")
        return

    input_file = 'data/synthetic_loan_data_rules.csv'

    if not os.path.exists(input_file):
        print(f"Error: Original data '{input_file}' not found.")
        print("Please run '01a_hands_on_statistical_rule_based.py' first.")
        return

    print("Loading original data to train GenAI model...")
    real_data = pd.read_csv(input_file)

    # ---------------------------------------------------------
    # STEP 1: Define Metadata
    # ---------------------------------------------------------
    # The GenAI model needs to know the data types of your columns
    print("Detecting metadata...")
    metadata = SingleTableMetadata()
    metadata.detect_from_dataframe(real_data)

    # ---------------------------------------------------------
    # STEP 2: Train the Generative AI Model (CTGAN)
    # ---------------------------------------------------------
    print("Initializing CTGAN Synthesizer...")
    # CTGAN (Conditional Tabular GAN) is great for mixed continuous/discrete data
    synthesizer = CTGANSynthesizer(
        metadata,
        epochs=50, # Keep low for workshop speed, increase for better quality
        verbose=True
    )

    print("Training the GAN on original data... (this may take a minute)")
    synthesizer.fit(real_data)

    # ---------------------------------------------------------
    # STEP 3: Sample New Synthetic Data
    # ---------------------------------------------------------
    print("Sampling 500 rows of AI-generated synthetic data...")
    synthetic_data = synthesizer.sample(num_rows=500)

    output_file = 'data/synthetic_loan_data_genai.csv'
    synthetic_data.to_csv(output_file, index=False)

    print(f"GenAI data successfully generated and saved to '{output_file}'")
    print("Sample GenAI Data:")
    print(synthetic_data.head())

if __name__ == "__main__":
    generate_genai_data()

Loading original data to train GenAI model...
Detecting metadata...
Initializing CTGAN Synthesizer...
Training the GAN on original data... (this may take a minute)


Gen. (+00.12) | Discrim. (-00.35): 100%|██████████| 50/50 [00:07<00:00,  6.69it/s]

Sampling 500 rows of AI-generated synthetic data...
GenAI data successfully generated and saved to 'data/synthetic_loan_data_genai.csv'
Sample GenAI Data:
   Age  Annual_Income  Credit_Score  Loan_Approved
0   35       11268.94           832              0
1   19       11268.94           727              1
2   23       17399.09           806              0
3   21       35269.90           682              1
4   22       11674.13           384              0
